# Building an Office Staff Agent with Vinagent

In this tutorial, we will build a powerful **Office Staff Agent** that can handle a full stack of office tools: **PowerPoint (pptx)**, **Word (docx)**, **Excel (xlsx)**, and **PDF**. 

By combining multiple specialized **AgentSkills**, we create a single cohesive agent capable of complex, cross-format workflows.

## 1. Setup and Initialization

Ensure you have a `.env` file with your `OPENAI_API_KEY`.

In [ ]:
# Install required dependencies if they are missing
# !pip install reportlab python-docx pdfplumber

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from vinagent.agent import Agent

load_dotenv(find_dotenv('.env'))

llm = ChatOpenAI(model="gpt-4o-mini")
print("LLM initialized.")

LLM initialized.


## 2. Loading the Multi-Skill Agent


In [ ]:
import os
# Get absolute path to the skills directory relative to this notebook
notebook_dir = os.getcwd()
base_skill_path = os.path.join(notebook_dir, "../../agentskills/skills/skills")
skills = ["xlsx", "docx", "pptx", "pdf"]
skill_paths = [os.path.normpath(os.path.join(base_skill_path, s)) for s in skills]

office_agent = Agent.load_agent(
    agent_paths=skill_paths, 
    llm=llm
)

print(f"Office Staff Agent loaded: {office_agent.name}")

INFO:vinagent.register.tool:Registered agentskill tool: 'xlsx' (working dir: D:\vinagent\agentskills\skills\skills\xlsx)
INFO:vinagent.register.tool:Registered agentskill tool: 'docx' (working dir: D:\vinagent\agentskills\skills\skills\docx)
INFO:vinagent.register.tool:Registered agentskill tool: 'pptx' (working dir: D:\vinagent\agentskills\skills\skills\pptx)
INFO:vinagent.register.tool:Registered agentskill tool: 'pdf' (working dir: D:\vinagent\agentskills\skills\skills\pdf)


Office Staff Agent loaded: xlsx_docx_pptx_pdf


## 3. Real-World Use Case Example

### Case 1: Data-Driven Presentation (Excel to PPT)

**Scenario:** You have raw sales data and need a pitch deck for the management team.

In [7]:
query1 = """
1. Create an Excel file 'quarterly_sales.xlsx' in the current directory with this data:
   Region | Sales | Growth
   North  | 500k  | 10%
   South  | 750k  | 25%
   West   | 400k  | -5%
2. Create a PowerPoint 'executive_summary.pptx' in the current directory with 2 slides:
   - Slide 1: Title 'Q1 Performance'
   - Slide 2: List the sales data from the Excel and mention South as growth leader.
"""

# Use Python for all tasks to avoid shell execution errors
response1 = office_agent.invoke(query1)
print(response1.content)

INFO:vinagent.agent.agent:No authentication card provided, skipping authentication
INFO:vinagent.agent.agent:I am chatting with unknown_user
INFO:vinagent.agent.agent:Tool calling iteration 1/10
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 8.776000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 18.160000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 4.076000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.logger.logger:tool_data: {
  "tool_name":

The Excel file 'quarterly_sales.xlsx' has been created successfully with the specified sales data, and the PowerPoint file 'executive_summary.pptx' has also been generated with the requested slides. You can find both files in the current directory.


### Case 2: Information Extraction (PDF to Excel & Word)

**Scenario:** You received a contract in PDF and need to extract key terms into a tracker and write a summary memo.

In [5]:
query2 = """
1. Create a dummy PDF 'contract_data.pdf' in the current directory with text: 'Effective Date: 2023-01-01, Total Value: $1,000,000, Vendor Name: Acme Corp'.
2. Read 'contract_data.pdf' and extract the 'Effective Date', 'Total Value', and 'Vendor Name'. Use pdfplumber if needed.
3. Save these fields into 'contract_tracker.xlsx' in the current directory.
4. Write a brief summary in 'contract_memo.docx' in the current directory.
"""

# Use Python for all tasks to avoid shell execution errors
response2 = office_agent.invoke(query2)
print(response2.content)

INFO:vinagent.agent.agent:No authentication card provided, skipping authentication
INFO:vinagent.agent.agent:I am chatting with unknown_user
INFO:vinagent.agent.agent:Tool calling iteration 1/10
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 16.307000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 8.518000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:vinagent.logger.logger:tool_data: {
  "tool_name": "pdf",
  "tool_type": "agentskills",
  "module_path": "D:\\vinagent\\agentskills\\skills\\skills\\pdf",
  "is_runtime": false,
  "tool_call_id": "pdf",
  "arguments": {
    "command": "fro

The PDF 'contract_data.pdf' was successfully created. The necessary information has been extracted and saved into 'contract_tracker.xlsx' and a brief summary has been written in 'contract_memo.docx'.


### Case 3: Monthly Reporting (Multiple PPTX & DOCX)

**Scenario:** Merging multiple department slides into one master deck and drafting a report.


**Note:** When merging multiple formats, the agent should use the respective skills (`pptx`, `docx`) sequentially to avoid environment mismatches.

In [6]:
query3 = """
1. Create two simple presentations in the current directory: 'marketing.pptx' (Slide: Marketing Goals) and 'engineering.pptx' (Slide: Engineering Milestones).
2. Merge them into 'monthly_report.pptx' in the current directory.
3. Create a Word doc 'monthly_summary.docx' in the current directory summarizing the achievements.
"""

# Use Python for all tasks to avoid shell execution errors
response3 = office_agent.invoke(query3)
print(response3.content)

INFO:vinagent.agent.agent:No authentication card provided, skipping authentication
INFO:vinagent.agent.agent:I am chatting with unknown_user
INFO:vinagent.agent.agent:Tool calling iteration 1/10
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 17.506000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 0.282000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying request to /chat/completions in 13.115000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:openai._base_client:Retrying reque

The presentations 'marketing.pptx' and 'engineering.pptx' have been created successfully. They were merged into 'monthly_report.pptx', and a summary Word document 'monthly_summary.docx' has also been created.
